In [ ]:
from pathlib import Path
import pandas as pd
import spatialdata as sd

import pickle
DATA_DIR = Path("/root/capsule/data")
SCRATCH_DIR = Path("/root/capsule/scratch")
MATCH_DIR = SCRATCH_DIR / "ophys_xenium_match_tables"
TRANSCRIPT_DIR = SCRATCH_DIR / 'transcriptomics'
TRANSCRIPT_DIR.mkdir(exist_ok=True)

load data information

In [2]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


In [5]:
for i_mouse, mouse_id in enumerate(mouse_ids):
    cell_types_df = pd.DataFrame()
    Xenium_mapped_path = DATA_DIR / data_info[mouse_id]['xenium']['cell_types']
    sections = [int(s.name.split('_')[1]) for s in Xenium_mapped_path.glob("section_*") if s.is_dir()]
    sections.sort()
    
    for section in sections:
        df = pd.read_csv(Xenium_mapped_path / f'section_{section}/mapped_data/basic_results.csv', skiprows=4)
        df['mouse_id'] = mouse_id
        df['xenium_section'] = section
        df = df[(df['subclass_bootstrapping_probability']>.6)]
        cell_types_df = pd.concat([cell_types_df, df])
    cell_types_df.rename(columns={'cell_id': 'cell_id - xenium'}, inplace=True)
    cell_types_df.to_csv(TRANSCRIPT_DIR / f'{mouse_id}_cell_types.csv', index=False)

In [6]:
for i_mouse, mouse_id in enumerate(mouse_ids):
    Xenium_processed_path = DATA_DIR / data_info[mouse_id]['xenium']['processed']
    sections = [int(s.name.split('_')[1].replace(".zarr","")) for s in Xenium_processed_path.glob("section_*") if s.is_dir()]
    sections.sort()
    cellxgene_df = []
    for section in sections:
    
        section_path = Xenium_processed_path / f'section_{section}.zarr'
        sdata = sd.read_zarr(str(section_path))

        df = sdata['table'].to_df().reset_index().rename(columns={'index':'cell_id - xenium'})
        df.insert(0, 'xenium_section', [section]*len(df))
        df.insert(0, 'mouse_id', [mouse_id]*len(df))
        cellxgene_df.append(df)
    cellxgene_df = pd.concat(cellxgene_df,axis=0)
    cellxgene_df.to_csv(TRANSCRIPT_DIR / f'{mouse_id}_cellxgene.csv', index=False)


KeyboardInterrupt: 

In [ ]:
main_subclasses = ['006 L4/5 IT CTX Glut',
'007 L2/3 IT CTX Glut',
'022 L5 ET CTX Glut',
'052 Pvalb Gaba',
'046 Vip Gaba',
'053 Sst Gaba',
'049 Lamp5 Gaba']
for i_mouse, mouse_id in enumerate(mouse_ids):
    ophys_xenium_match = pd.read_csv(MATCH_DIR / f'{mouse_id}_ophys_xenium_match.csv')
    cell_types_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}_cell_types.csv')
    cell_types_coreg_df = ophys_xenium_match.merge(cell_types_df, on = ['mouse_id','xenium_section','cell_id - xenium'], how = 'left')
    cell_types_coreg_df = cell_types_coreg_df[cell_types_coreg_df['subclass_name'].isin(main_subclasses)]
    cell_types_coreg_df.to_csv(TRANSCRIPT_DIR / f'{mouse_id}_cell_types_coreg.csv',index=False)

    cellxgene_coreg_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}_cellxgene_coreg.csv')
    cellxgene_coreg_df = ophys_xenium_match.merge(cellxgene_df, on = ['mouse_id','xenium_section','cell_id - xenium'], how = 'left')
    cellxgene_coreg_df = cellxgene_coreg_df[cellxgene_coreg_df['unique_id'].isin(cell_types_coreg_df['unique_id'].values)]
    cellxgene_coreg_df = cellxgene_coreg_df[[col for col in cellxgene_coreg_df.columns if 'Code' not in col and 'Control' not in col]]
    cellxgene_coreg_df.to_csv(TRANSCRIPT_DIR / f'{mouse_id}_cellxgene_coreg.csv',index=False)
